# Prática — Módulo 10 - Word2Vec + KMeans

In [20]:
# Carregar dados
import pandas as pd

df = pd.read_parquet("/content/df_final_1024.parquet")

print("Shape:", df.shape)

Shape: (736, 10)


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736 entries, 0 to 735
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ClienteCod             736 non-null    int64 
 1   ConteudoCategoriaNome  736 non-null    object
 2   ConteudoNome           736 non-null    object
 3   ConteudoDescricao      736 non-null    object
 4   qtd_conteudos          736 non-null    int64 
 5   nome_limpo             736 non-null    object
 6   desc_limpa             736 non-null    object
 7   categorias_limpas      736 non-null    object
 8   texto_final            736 non-null    object
 9   cluster                736 non-null    int32 
dtypes: int32(1), int64(2), object(7)
memory usage: 54.8+ KB


In [22]:
# Criar campo textual único para vetorização
# Determinar qual campo textual vale manter
df["texto_final"] = (
    df["categorias_limpas"].apply(lambda x: " ".join(x))  # <- lista de palavras
    # + " " + df["nome_limpo"]
    # + " " + df["desc_limpa"]
).str.strip()

# **Fase 0 - Engenharia de Atributos**

In [23]:
# Transformar cada texto em lista de tokens
import re

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-záéíóúâêôãõç\s]", " ", text)
    return text.split()

df["tokens"] = df["texto_final"].apply(tokenize)

# **Fase 1 - Representação Word2Vec**

In [24]:
# Treinar embeddings de palavras com Word2Vec
!pip install gensim
from gensim.models import Word2Vec

sentences = df["tokens"].tolist()

w2v_model = Word2Vec(
    sentences=sentences,      # lista de sentenças (par de treinamento)
    vector_size=100,          # dimensão do embedding (tamanho do vetor de cada palavra)
    window=5,                 # tamanho da janela de contexto (palavras ao redor consideradas)
    min_count=2,              # ignora palavras com frequência < 2 (remove termos raros)
    workers=4                 # número de threads usadas no treino (paralelismo)
)

## Pergunta
1. Qual foi o parâmetro vector_size que usamos no exemplo da aula?

In [25]:
# Criar IDF
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
vectorizer.fit(df["texto_final"])

idf_dict = dict(zip(
    vectorizer.get_feature_names_out(),
    vectorizer.idf_
))

In [26]:
# Representar cada conteúdo pela média dos vetores das palavras
import numpy as np

def doc_vector(tokens):
    vectors = []
    weights = []

    for w in tokens:
        if w in w2v_model.wv and w in idf_dict:
            vectors.append(w2v_model.wv[w])
            weights.append(idf_dict[w])

    if not vectors:
        return np.zeros(100)

    return np.average(vectors, axis=0, weights=weights)

X = np.vstack(df["tokens"].apply(doc_vector))

print("Shape:", X.shape)

Shape: (736, 100)


## Pergunta
1. Após o Word2Vec, o que exatamente temos como saída?  
Resposta: Um vetor numérico para cada palavra do vocabulário (não para o documento).

2. Pergunta: Por que precisamos calcular a média dos vetores das palavras (com ou sem IDF)?   
Resposta: Para gerar um único vetor que represente o documento inteiro, já que o Word2Vec só gera vetores por palavra. É um hack que se usava antigamente.

# **Fase 2 - Clustering (KMeans)**

In [27]:
# Agrupar conteúdos usando KMeans
from sklearn.cluster import KMeans

k = 32

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

## Validação

### Silhouette Score

In [28]:
# Calcular qualidade geométrica dos clusters
from sklearn.metrics import silhouette_score

score = silhouette_score(X, df["cluster"])
print("Silhouette:", score)

Silhouette: 0.4880630302416178


### Visualização de Clusters

In [29]:
# Ver top termos por cluster
from sklearn.metrics.pairwise import cosine_similarity

for i in range(k):
    centroid = kmeans.cluster_centers_[i]

    sims = cosine_similarity([centroid], w2v_model.wv.vectors)[0]
    top_idx = sims.argsort()[-10:]

    terms = w2v_model.wv.index_to_key
    print(f"\nCluster {i}:")
    print(", ".join([terms[j] for j in top_idx]))


Cluster 0:
para, ferramentas, de, medicina, desenv, pessoal, saúde, gestão, e, educação

Cluster 1:
médica, e, redes, ti, educação, para, consultoria, marketing, gestão, vendas

Cluster 2:
profissional, de, ferramentas, pessoal, para, desenv, educação, gestão, e, marketing

Cluster 3:
beleza, ferramentas, para, saúde, desenv, pessoal, de, educação, gestão, e

Cluster 4:
de, saúde, qualidade, medicina, para, desenv, pessoal, gestão, e, educação

Cluster 5:
saúde, qualidade, investimentos, psicologia, e, gestão, medicina, ferramentas, educação, games

Cluster 6:
negócios, arte, beleza, e, cursos, para, criatividade, agronegócio, ferramentas, enfermagem

Cluster 7:
iniciação, saúde, desenv, educação, ferramentas, de, gestão, e, desenvolvimento, pessoal

Cluster 8:
para, pessoal, ferramentas, desenv, de, negócios, administração, educação, gestão, e

Cluster 9:
para, dieta, felicidade, educação, pessoal, gestão, de, e, saúde, beleza

Cluster 10:
marketing, desenv, cursos, felicidade, saúde

In [30]:
# Ver exemplos de separação
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(
    df.groupby("cluster")["texto_final"]
    .apply(lambda x: x.sample(min(len(x), 4), random_state=42))
)

cluster     
0        527                                                                                          saúde educação
         548                                                                                    educação e carreiras
         15                                                                                     educação e carreiras
         569                                                                                          saúde educação
1        274                                                                                                  vendas
         624                                                                                                  vendas
         589                                                                                                  vendas
         485                                                                                                  vendas
2        483                                                    gastronomia propaganda e marketing marketing digital
         105                                                                 marketing arquitetura marketing digital
         321                                                                                        marketing vendas
         16                                                                  marketing odontologia marketing digital
3        27                               vendas gestão e liderança empreendedorismo produtividade marketing digital
         346                                        marketing redes sociais propaganda e marketing marketing digital
         74       empreendedorismo negócios e dinheiro investimentos autoajuda e desenv. pessoal educação financeira
         6                                                                     saúde medicina alternativa psicologia
4        577                                                                                                educação
         282                                                                                                educação
         505                                                                                                educação
         476                                                                                                educação
5        94                                                                                               3d e games
         201                                                                                              3d e games
         647                                                                                              3d e games
         306                                                                                              3d e games
6        100                                                                                             arquitetura
         107                                                                                   música e instrumentos
         305                                                                                   direito contabilidade
         663                                                                                              enfermagem
7        408                                                                                 desenvolvimento pessoal
         585                                                                                 desenvolvimento pessoal
         44                                                                                  desenvolvimento pessoal
         617                                                                                 desenvolvimento pessoal
8        676                                                                                                medicina
         320                                                                                administração e negócios
         

## Pergunta
4. O Silhouette Score obtido e a sua percepção como analista agora estão compatíveis?